In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e3_path = kagglehub.competition_download('playground-series-s6e3')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [ ]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Load data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# Drop ID immediately to avoid using it as a feature
train_id = train.pop('id')
test_id = test.pop('id')

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

# Initial Look
display(train.head())
display(train.describe())

<h2>2. The "Target" Check</h2>

In [ ]:
# Target Distribution
target = 'Churn'

print(f"\n--- Target Distribution ({target}) ---")
print(train[target].value_counts())

print(train[target].value_counts(normalize=True))

sns.countplot(data=train, x=target)
plt.title("Target Distribution")
plt.show()

<h2>3. Cardinality & Missing Values</h2>

In [ ]:
# Missing Values & Cardinality
eda_summary = pd.DataFrame({
    'Dtype': train.dtypes,
    'Nulls': train.isnull().sum(),
    'Unique': train.nunique(),
}).sort_values(by='Unique')

display(eda_summary)

<h2>4. Bivariate Analysis & Leakage Hunting</h2>

In [ ]:
# 1. Numerical Features vs Target (KDE Plots)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
plt.figure(figsize=(15, 5))

for i, col in enumerate(num_cols):
    plt.subplot(1, 3, i+1)
    sns.kdeplot(data=train, x=col, hue='Churn', fill=True, common_norm=False)
    plt.title(f'{col} Distribution by Churn')

plt.tight_layout()
plt.show()

# 2. Categorical Features vs Target (Churn Rate per Category)
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # Calculate churn rate per category
    churn_rate = train.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values()
    sns.barplot(x=churn_rate.index, y=churn_rate.values, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Churn Rate by {col}')
    axes[i].set_ylabel('Proportion of Churn')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Based on plots, we’ve identified the "High-Risk Profile":

**The Contract Trap**: Month-to-month customers have a churn rate over 40%, while long-term contracts are incredibly stable. This is your strongest predictor.

**The Fiber Optic Mystery**: Usually, faster internet means happier customers, but here, Fiber optic users are churning at a massive rate (~40%). This suggests a service quality issue or a high price point in the synthetic data.

**The Payment Method Artifact**: Electronic check is a huge outlier. Nearly 50% of these customers leave. Automatic payments (Credit Card/Bank Transfer) are much safer.

**The Tenure "Danger Zone"**: Churn is concentrated in the first 0–10 months. If a customer survives the first year, they are likely to stay forever.

<h2>5. Feature Engineering</h2>

Since we found three "Danger Zones" (Month-to-month, Fiber optic, Electronic check), let's create a feature that counts how many "red flags" a customer has.

In [ ]:
# Create a simple risk counter
train['Risk_Count'] = (
    (train['Contract'] == 'Month-to-month').astype(int) +
    (train['InternetService'] == 'Fiber optic').astype(int) +
    (train['PaymentMethod'] == 'Electronic check').astype(int)
)

Tenure Binning
Since tenure isn't linear (the difference between month 1 and 2 is huge, but month 60 and 61 is negligible), we should group them.
* New: 0-12 months
* Established: 13-48 months
* Loyal: 49+ months

Financial Ratios


* Monthly_per_Total: MonthlyCharges / (TotalCharges + 1) (How much of their total spend happened this month?)
* Tenure_per_Total: TotalCharges / (tenure + 1) (Verify if this matches MonthlyCharges or if there's a "discount" hidden in the gap).


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Feature Engineering (The 'Grandmaster' additions)
for df in [train, test]:
    # Financial features
    df['Monthly_per_Total'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['Tenure_Risk'] = df['tenure'].apply(lambda x: 1 if x < 12 else 0)

    # Combined Risk Feature
    df['Is_High_Risk_Combo'] = ((df['Contract'] == 'Month-to-month') &
                                (df['InternetService'] == 'Fiber optic') &
                                (df['PaymentMethod'] == 'Electronic check')).astype(int)

# 2. Encoding Categorical Variables
# We'll use Label Encoding for simplicity in our baseline
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in cat_cols: cat_cols.remove('Churn')

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

# Convert Target to Numeric
train['Churn'] = train['Churn'].map({'No': 0, 'Yes': 1})

print("Preprocessing Complete. Features ready for modeling.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate correlations with the target
correlations = train.corr()['Churn'].sort_values(ascending=False)

# Plotting the correlations
plt.figure(figsize=(10, 8))
correlations.drop('Churn').plot(kind='barh', color='skyblue')
plt.title('Feature Correlation with Churn (Target)')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Features')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Print the top 10 most correlated features
print("--- Top 10 Features Correlated with Churn ---")
print(correlations.abs().sort_values(ascending=False).head(11))

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Selecting a subset for speed if needed, but 600k is manageable
X = train.drop('Churn', axis=1)
y = train['Churn']

# Calculate Mutual Information (this may take a minute)
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, name="MI Scores", index=X.columns).sort_values(ascending=False)

print("\n--- Top Mutual Information Scores ---")
print(mi_series.head(10))

In [ ]:
# 1. Re-apply the Feature Engineering to BOTH (to be 100% sure)
for df in [train, test]:
    # Financial features
    df['Monthly_per_Total'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['Tenure_Risk'] = (df['tenure'] < 12).astype(int)

    # Risk features
    df['Risk_Count'] = (
        (df['Contract'] == 'Month-to-month').astype(int) +
        (df['InternetService'] == 'Fiber optic').astype(int) +
        (df['PaymentMethod'] == 'Electronic check').astype(int)
    )

    df['Is_High_Risk_Combo'] = ((df['Contract'] == 'Month-to-month') &
                                (df['InternetService'] == 'Fiber optic') &
                                (df['PaymentMethod'] == 'Electronic check')).astype(int)

# 2. Re-define X and X_test
X = train.drop('Churn', axis=1)
y = train['Churn']

# This line is the magic trick: it ensures X_test has the same cols in the same order as X
X_test = test[X.columns]

print(f"Alignment Check: Train shape {X.shape}, Test shape {X_test.shape}")

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# 1. Prepare Data
X = train.drop('Churn', axis=1)
y = train['Churn']
X_test = test.copy()

# 2. Hyperparameters (Grandmaster Baseline)
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'n_estimators': 10000, # Large number with early stopping
    'n_jobs': -1,
    'random_state': 42,
    'verbosity': -1
}

# 3. Cross-Validation Loop
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
feature_importance_df = pd.DataFrame()

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Training Fold {fold + 1} ---")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(200)]
    )

    # Predict Out-of-Fold and Test set
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / kf.n_splits

    # Track Feature Importance
    fold_importance = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
    feature_importance_df = pd.concat([feature_importance_df, fold_importance], axis=0)

# Final CV Score
overall_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall CV ROC-AUC: {overall_auc:.5f}")

In [ ]:
# Visualize Feature Importance
best_features = feature_importance_df.groupby('feature')['importance'].mean().sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=best_features.values, y=best_features.index, palette='magma')
plt.title('Top 15 Features by LightGBM Importance')
plt.show()

# Create Submission File
submission = pd.DataFrame({
    'id': test_id,
    'Churn': test_preds
})
submission.to_csv('submission.csv', index=False)
print("Submission file saved as submission.csv")